In [ ]:
!pip install transformers accelerate peft datasets sentencepiece bitsandbytes

import pandas as pd
import torch, gc
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, pipeline, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training

df = pd.read_csv("/content/generated_qa_135.csv")  

print(f"Loaded {len(df)} training examples")
print(df.head())

dataset = Dataset.from_pandas(df)

gc.collect()
torch.cuda.empty_cache()

model_name = "tiiuae/falcon-rw-1b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)


model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize_function(examples):
    texts = []

    for p, c in zip(examples["prompt"], examples["completion"]):
        if "Antwort:" in c:
            answer = c.split("Antwort:")[-1].strip()
        else:
            answer = c.strip()

        text = f"Frage: {p}\nAntwort: {answer}"
        texts.append(text)

    tokenized = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, batch_size=16)

gc.collect()
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./tax-lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    num_train_epochs=1,   
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)


trainer.train()


model.save_pretrained("./tax-lora")
tokenizer.save_pretrained("./tax-lora")


gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16
)

model = PeftModel.from_pretrained(model, "./tax-lora")
tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0,
    max_new_tokens=150,
    do_sample=False
)

original_df = pd.read_csv("/content/dataset_clean.csv")

answers = []

for i, row in original_df.iterrows():
    prompt_text = f"Frage: {row['prompt']}\nAntwort:"
    output = generator(prompt_text)[0]["generated_text"]

    answer = output.split("Antwort:")[-1].strip()
    answers.append(answer)

    print(f"{i+1}/{len(original_df)}")

results_df = pd.DataFrame({
    "id": original_df["id"],
    "answer": answers
})

results_df.to_csv("final_results.csv", index=False)